# Flux LoRA Train

⚠️ **Remember to copy this notebook in your Drive and rename.**


**This workflow uses AI Toolkit'**

[jhj0517/finetuning-notebooks](https://github.com/jhj0517/finetuning-notebooks)

[ostris/ai-toolkit](https://github.com/ostris/ai-toolkit)

**HuggingFace Alternative**

[README_flux.md](https://github.com/huggingface/diffusers/blob/main/examples/dreambooth/README_flux.md)

[train_dreambooth_lora_flux.py](https://github.com/huggingface/diffusers/blob/main/examples/dreambooth/train_dreambooth_lora_flux.py)

[test_dreambooth_lora_flux.py](https://github.com/huggingface/diffusers/blob/main/examples/dreambooth/test_dreambooth_lora_flux.py)

*Workflows for IAAC MaCDA GenAI  (Apr - Jun 2026) taught by [James McBennett](https://www.linkedin.com/in/mcbennett/) and [Aymeric Brouez](https://www.linkedin.com/in/aymeric-brouez/)*

*With special thanks to past faculty [Nono Martínez Alonso](https://youtube.com/NonoMartinezAlonso).*

##Confirm using A100 GPU

You absolutely need an A100 to train a Flux LoRA. Sometimes Colab says that it is connected to an A100 while secretly connecting to a lower GPU. Confirm below that you actually connected to an A100 as this isn't going to work if you are connected to a lesser GPU.

In [1]:
!nvidia-smi

Fri Jun 12 08:43:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   35C    P0             56W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## Install

In [2]:
!git clone -b fix/numpy-error https://github.com/jhj0517/ai-toolkit.git
%cd ai-toolkit
!git submodule update --init --recursive > /dev/null 2>&1
!pip install --quiet -r requirements.txt

#Downgrade Numpy
!pip uninstall -y numpy
!pip install --quiet --force-reinstall --no-deps numpy==1.26.3

Cloning into 'ai-toolkit'...
remote: Enumerating objects: 5576, done.
remote: Total 5576 (delta 0), reused 0 (delta 0), pack-reused 5576 (from 1)
Receiving objects: 100% (5576/5576), 30.64 MiB | 40.96 MiB/s, done.
Resolving deltas: 100% (3893/3893), done.
/content/ai-toolkit
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.5/96.5 kB 10.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.4/202.4 kB 5.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 766.6/766.6 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 67.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# Restart session
import os
os.kill(os.getpid(), 9)
print("IGNORE: 'Your session crashed for an unknown reason'. This was done intentionally.")
# Continue to the next cell and keep going after runtime restarts

## Setup

In [1]:
# Fix numpy incompatibility from https://github.com/ostris/ai-toolkit/issues/267
import numpy
print("Numpy should be 1.26.3. The version currently being used is: " + numpy.__version__)

Numpy should be 1.26.3. The version currently being used is: 1.26.3


## Mount Drive

In [2]:
from google.colab import drive
import os
drive.mount('/content/drive')

Mounted at /content/drive


## Hugging Face Token

In [3]:
# Sign up at Hugging Face and create a "Read" access token (not the default "Fine-Grained" token).
# Click the 🔑 "Secrets" icon in the left sidebar.
# Enable Notebook Access, Set the Name to "HF_TOKEN", Paste your token as the Value

from google.colab import userdata
hf_token = userdata.get("HF_TOKEN")

## Set Directories

In [4]:
DATASET_DIR = '/content/drive/MyDrive/cartoonify/02_FLUX.1/dataset_FLUX.1'
OUTPUT_DIR = '/content/drive/MyDrive/cartoonify/02_FLUX.1/weights_FLUX.1'
LORA_NAME = 'gdo_cartoon'
os.makedirs(OUTPUT_DIR, exist_ok=True)

## Login to Weights & Biases with token

In [5]:
# Get your API token from https://wandb.ai/settings
# Click the empty space after "quit:" to enter code
!pip install wandb -q
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: symonkipkemei (symonkipkemei-iaac) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## Config

In [6]:
import sys
sys.path.append('/content/ai-toolkit')
from toolkit.job import run_job
from collections import OrderedDict
from PIL import Image
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

In [7]:
# Model
repo_id_or_path = 'black-forest-labs/FLUX.1-dev'
quantize = False
dtype = "bf16" #"float16"

# Training Steps (max_steps)
# More steps = more learning, higher risk of overfitting
# Fewer steps = less learning, may underfit
# 1000  = quick test / smoke run
# 1500  = good for small datasets with low LR
# 2000  = default starting point (30 images)
# 3000  = larger datasets or lower LR runs
# 4000  = high variety datasets, risk of overfit on small sets
steps = 2000

# Saving checkpoints
# Saves a checkpoint every save_every steps
# Deletes oldest checkpoint once max_step_saves_to_keep is exceeded
save_every = 250
max_step_saves_to_keep = 10

# Sampling
sample_every = 250
sample_seed = 77
sample_steps = 50
width = 1024
height = 1024

# Learning Rate
# Higher = faster learning, more risk of overfitting
# Lower  = slower learning, more stable convergence
# 4e-4 = 0.0004  (FLUX.1 default, aggressive)
# 2e-4 = 0.0002  (moderate)
# 1e-4 = 0.0001  (conservative, recommended for FLUX.2)
# 5e-5 = 0.00005 (slow, stable, good for small datasets)
# 1e-5 = 0.00001 (very slow, use if overfitting persists)
lr = 4e-4

# Replace Prompts with your own first three or favorite prompts
sample_prompt_1 = "gdo_cartoon editorial cartoon | political cartoon | caricature | newspaper illustration, pen and ink | cross-hatching | bold outlines | hand-drawn linework | varied line weight | gestural strokes, black and white | monochrome | print cartoon | white background | no color fill, satirical | confrontational | exaggerated | humorous tension | absurdist, dry wit | political commentary | ironic | sharp | deadpan, three figure layout | horizontal spread | left to right narrative flow | large negative white space | frontal view | eye level | speech bubble top left | asymmetric balance | isolated figures | implied ground plane"
sample_prompt_2 = "gdo_cartoon editorial cartoon | political cartoon | caricature | newspaper illustration, pen and ink | cross-hatching | bold outlines | hand-drawn linework | spot color red | varied line weight | dense detail, black white and red | limited color | print cartoon | white background, dark | disturbing | satirical | grotesque | critical, scathing | accusatory | macabre | political condemnation | darkly ironic, vertical layering | figure above chaos | foreground to background depth | desk as stage | crowd below | top-heavy weight | speech bubble top right | birdcage prop | portrait inset top left | dense lower third"
sample_prompt_3 = "gdo_cartoon editorial cartoon | political cartoon | caricature | strip format | multi-panel, pen and ink | cross-hatching | bold outlines | hand-drawn linework | decorative pattern fills | varied line weight, black and white | monochrome | print cartoon | paneled layout, satirical | skeptical | cynical | dry humour | wry, political commentary | ironic | deadpan | comparative | resigned, three panel horizontal strip | portrait frames within panels | figures below portraits | left to right narrative | speech bubbles per panel | bordered panels | equal panel width | eye level | frontal figures"
sample_prompts = [sample_prompt_1, sample_prompt_2, sample_prompt_3]


# Advanced
# Lora Network
linear = 16
linear_alpha = 16

# Dataset
caption_ext = "txt"
caption_dropout_rate = 0.05
shuffle_tokens = False
cache_latents_to_disk = True
resolution = [1024]

# Training
performance_log_every = 1000
train_only_specific_layers = False
only_if_contains = ["transformer.single_transformer_blocks.7.proj_out", "transformer.single_transformer_blocks.20.proj_out"]
batch_size = 1
gradient_accumulation_steps = 1
train_dtype = "bf16"
train_unet = True
train_text_encoder = True
content_or_style = 'balanced'
gradient_checkpointing = True
noise_scheduler = 'flowmatch'
optimizer = 'adamw8bit'
use_ema = True
ema_decay = 0.99

## Train

In [8]:
job_to_run = OrderedDict([
    ('job', 'extension'),
    ('config', OrderedDict([
        ('name', LORA_NAME),
        ('process', [
            OrderedDict([
                ('type', 'sd_trainer'),
                ('training_folder', OUTPUT_DIR),
                ('performance_log_every', 1000),
                ('device', 'cuda:0'),
                ('network', OrderedDict([
                    ('type', 'lora'),
                    ('linear', linear),
                    ('linear_alpha', linear_alpha),
                    ('network_kwargs', OrderedDict([
                      ("prefix", [
                          "lora_te1.text_model.encoder",
                          "transformer.single_transformer_blocks",
                          "transformer.transformer_blocks"
                      ])
                    ]))
                ])),
                ('save', OrderedDict([
                    ('dtype', dtype),
                    ('save_every', save_every),
                    ('max_step_saves_to_keep', max_step_saves_to_keep)
                ])),
                ('datasets', [
                    OrderedDict([
                        ('folder_path', DATASET_DIR),
                        ('caption_ext', caption_ext),
                        ('caption_dropout_rate', caption_dropout_rate),
                        ('shuffle_tokens', shuffle_tokens),
                        ('cache_latents_to_disk', cache_latents_to_disk),
                        ('resolution', resolution)
                    ])
                ]),
                ('train', OrderedDict([
                    ('batch_size', batch_size),
                    ('steps', steps),
                    ('gradient_accumulation_steps', gradient_accumulation_steps),
                    ('train_unet', train_unet),
                    ('train_text_encoder', train_text_encoder),
                    ('content_or_style', content_or_style),
                    ('gradient_checkpointing', gradient_checkpointing),
                    ('noise_scheduler', noise_scheduler),
                    ('optimizer', optimizer),
                    ('lr', lr),
                    ('ema_config', OrderedDict([
                        ('use_ema', use_ema),
                        ('ema_decay', ema_decay)
                    ])),
                    ('dtype', train_dtype)
                ])),
                ('model', OrderedDict([
                    ('name_or_path', repo_id_or_path),
                    ('is_flux', True),
                    ('quantize', quantize),
                ])),
                ('sample', OrderedDict([
                    ('sampler', 'flowmatch'),
                    ('sample_every', sample_every),
                    ('width', width),
                    ('height', height),
                    ('prompts', sample_prompts),
                    ('neg', ''),
                    ('seed', sample_seed),
                    ('walk_seed', True),
                    ('guidance_scale', 4),
                    ('sample_steps', sample_steps)
                ]))
            ])
        ])
    ])),
    ('meta', OrderedDict([
        ('name', '[name]'),
        ('version', '1.0')
    ]))
])

# Conditional Parameters
if train_only_specific_layers:
    network = job_to_run["config"]["process"][0]["network"]
    network_kwargs = network.setdefault("network_kwargs", OrderedDict())
    network_kwargs["only_if_contains"] = only_if_contains


wandb.init(project='flux-lora', name=LORA_NAME, resume='allow')
run_job(job_to_run)

/usr/local/lib/python3.12/dist-packages/albumentations/__init__.py:13: UserWarning: A new version of Albumentations is available: 2.0.8 (you have 1.4.15). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()
/usr/local/lib/python3.12/dist-packages/controlnet_aux/mediapipe_face/mediapipe_face_common.py:7: UserWarning: The module 'mediapipe' is not installed. The package will have limited functionality. Please install it using the command: pip install 'mediapipe'
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/usr/local/lib/python3.12/dist-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated

{
    "type": "sd_trainer",
    "training_folder": "/content/drive/MyDrive/cartoonify/02_FLUX.1/weights_FLUX.1",
    "performance_log_every": 1000,
    "device": "cuda:0",
    "network": {
        "type": "lora",
        "linear": 16,
        "linear_alpha": 16,
        "network_kwargs": {
            "prefix": [
                "lora_te1.text_model.encoder",
                "transformer.single_transformer_blocks",
                "transformer.transformer_blocks"
            ]
        }
    },
    "save": {
        "dtype": "bf16",
        "save_every": 250,
        "max_step_saves_to_keep": 10
    },
    "datasets": [
        {
            "folder_path": "/content/drive/MyDrive/cartoonify/02_FLUX.1/dataset_FLUX.1",
            "caption_ext": "txt",
            "caption_dropout_rate": 0.05,
            "shuffle_tokens": false,
            "cache_latents_to_disk": true,
            "resolution": [
                1024
            ]
        }
    ],
    "train": {
        "batch_size": 1

config.json:   0%|          | 0.00/378 [00:00<?, ?B/s]

(…)ion_pytorch_model.safetensors.index.json: 0.00B [00:00, ?B/s]

transformer/diffusion_pytorch_model-0000(…):   0%|          | 0.00/9.98G [00:00<?, ?B/s]

...

Saving at step 1500
Saved to /content/drive/MyDrive/cartoonify/02_FLUX.1/weights_FLUX.1/gdo_cartoon/optimizer.pt


Generating Images: 100%|██████████| 3/3 [01:15<00:00, 25.11s/it]
                                                                

Saving at step 1750
Saved to /content/drive/MyDrive/cartoonify/02_FLUX.1/weights_FLUX.1/gdo_cartoon/optimizer.pt


gdo_cartoon: 100%|█████████▉| 1999/2000 [1:27:58<00:02,  2.64s/it, lr: 4.0e-04 loss: 1.946e-01]



Saved to /content/drive/MyDrive/cartoonify/02_FLUX.1/weights_FLUX.1/gdo_cartoon/optimizer.pt


## Disconnect

In [ ]:
from google.colab import runtime
runtime.unassign()